In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)  # 72

72

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]



In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import llm_structured, calc_price

load_dotenv()               # читает OPENAI_API_KEY из .env
openai_client = OpenAI()

In [6]:
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

2026-07-13 16:25:28.087230061 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [7]:
first_pages = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
by_name = {d["filename"]: d for d in documents}
pages = [by_name[name] for name in first_pages]

usages = []
gen_records = []
for doc in pages:
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    result, usage = llm_structured(
        openai_client, data_gen_instructions, user_prompt, Questions,
        model="gpt-5.4-mini",
    )
    usages.append(usage)
    for q in result.questions:
        gen_records.append({"question": q, "filename": doc["filename"]})
    print(doc["filename"], "-> input_tokens:", usage.input_tokens)

01-agentic-rag/lessons/01-intro.md -> input_tokens: 1021
01-agentic-rag/lessons/02-environment.md -> input_tokens: 1287
01-agentic-rag/lessons/03-rag.md -> input_tokens: 1754


In [8]:
avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
print("Q1. average input tokens:", avg_input_tokens)
# порядок величины ~ 1400 -> ближайший вариант 1400

Q1. average input tokens: 1354.0


In [9]:
ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
len(ground_truth), ground_truth[0]

(360,
 {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'})

In [10]:
pd.read_csv("ground-truth.csv").head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [11]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)  # 295

295

In [12]:
# текстовый (keyword) поиск
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

def text_search(query, num_results=5):
    return tindex.search(query, num_results=num_results)

In [13]:
# векторный поиск на локальных ONNX-эмбеддингах (all-MiniLM-L6-v2)
from embedder import Embedder
embed = Embedder()

texts = [c["content"] for c in chunks]
X = []
batch_size = 50
for i in tqdm(range(0, len(texts), batch_size), desc="embedding chunks"):
    X.extend(embed.encode_batch(texts[i:i + batch_size]))
X = np.array(X)

vindex = VectorSearch()
vindex.fit(X, chunks)

def vector_search(query, num_results=5):
    q = embed.encode(query)
    return vindex.search(q, num_results=num_results)

embedding chunks:   0%|          | 0/6 [00:00<?, ?it/s]

In [14]:
# гибридный поиск: RRF-слияние текстового и векторного (как в ДЗ 2)
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [15]:
q = ground_truth[0]["question"]
print(q)
text_search(q)[0]["filename"]

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


'01-agentic-rag/lessons/03-rag.md'

In [16]:
vector_search(q)[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [17]:
def compute_relevance(q, search_function):
    fname = q["filename"]
    results = search_function(q["question"])
    return [int(d["filename"] == fname) for d in results]

def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]

def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)

def mrr(relevance):
    total = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total += 1 / (rank + 1)
                break
    return total / len(relevance)

def evaluate(ground_truth, search_function):
    relevance = compute_relevance_total(ground_truth, search_function)
    return {"hit_rate": hit_rate(relevance), "mrr": mrr(relevance)}

In [18]:
res_text = evaluate(ground_truth, text_search)
res_text  # hit_rate ~ 0.758 -> ближайший 0.76

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [19]:
for k in [1, 50, 100, 200]:
    res = evaluate(ground_truth, lambda query: hybrid_search(query, k=k))
    print(f"k={k:>3}: hit_rate={res['hit_rate']:.4f}  mrr={res['mrr']:.4f}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=  1: hit_rate=0.8389  mrr=0.6482


  0%|          | 0/360 [00:00<?, ?it/s]

k= 50: hit_rate=0.8361  mrr=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: hit_rate=0.8361  mrr=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: hit_rate=0.8361  mrr=0.6379
